In [2]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
import utils.file_IO as file_IO
import pandas as pd

# are preprocessed files ok?! Yes

In [3]:
source_file_train='icdar_train_df_patches_20250716_113702'
extra_file_train = 'icdar_train_df_body_20250523_181312'
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{source_file_train}.csv")
train_df_2 = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{extra_file_train}.csv")

In [4]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio']


In [5]:
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2']


In [7]:
train_df_2['file_name'][0]

'C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset\\unzipped\\1_50\\0001_1.jpg'

In [10]:
train_df=file_IO.change_filename_from_to(train_df, fr='old-laptop', to='new-laptop')

In [11]:
train_df['file_name'][0]

'C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset\\unzipped\\1_50\\0001_1.jpg'

In [12]:
combined = pd.concat([train_df[['writer', 'isEng', 'same_text']],
                            train_df_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(combined))

1128


In [13]:
# Merge group ids back to each original DataFrame
train_df = train_df.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_df_2 = train_df_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')

In [14]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
train_df[cols_to_drop].columns

Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page'],
      dtype='object')

In [15]:
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
train_df_2[cols_to_drop].columns

Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'page'],
      dtype='object')

# are extracted files ok?! yes, error was in the page column being already present

In [3]:
train=r"c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14\representation_extraction\extracted_representation\train\clip-vit-large-patch14_features_icdar_train_df_patches_20250716_113702.csv"
extra=r"c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14\representation_extraction\extracted_representation\extra_view\train\clip-vit-large-patch14_features_icdar_train_df_body_20250523_181312.csv"

In [25]:
train_df = pd.read_csv(train)
train_df_2 = pd.read_csv(extra)

In [26]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page']
Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'page']


In [27]:
train_df.drop(columns=['page'], inplace=True)
train_df_2.drop(columns=['page'], inplace=True)

In [28]:
combined = pd.concat([train_df[['writer', 'isEng', 'same_text']],
                            train_df_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(combined))
# Merge group ids back to each original DataFrame
train_df = train_df.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_df_2 = train_df_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(train_df[cols_to_drop].columns)
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(train_df_2[cols_to_drop].columns)

1128
Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page'],
      dtype='object')
Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'page'],
      dtype='object')


# is the merging ok? cause i have troubles in the ensemble vs individual accuracy

In [34]:
train_1 = pd.read_csv(train)
train_2 = pd.read_csv(extra)
train_1.drop(columns=['page'], inplace=True) if 'page' in train_1.columns else None
train_2.drop(columns=['page'], inplace=True) if 'page' in train_2.columns else None
# Concatenate both datasets to build a unified group mapping
combined = pd.concat([train_1[['writer', 'isEng', 'same_text']],
                    train_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
# Merge group ids back to each original DataFrame
train_1 = train_1.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_2 = train_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
cols_to_drop_1 = [c for c in train_1.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
cols_to_drop_2 = [c for c in train_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
#print(train_2[cols_to_drop_2].head())
common_cols = list(set(cols_to_drop_1) & set(cols_to_drop_2))
# Remove 'page' from common_cols if present
if 'page' in common_cols:
    common_cols.remove('page')
    common_cols.remove('file_name')  # Assuming 'file_name' is not needed for merging

In [35]:
train_2.drop(columns=common_cols, inplace=True, errors='ignore')
num_pages = combined['page'].nunique()
patch_1_per_page = int(len(train_1) / num_pages)
patch_2_per_page = int(len(train_2) / num_pages)
# Repeat train_2 so it matches the number of patches per page in train_1
repeat_factor = patch_1_per_page // patch_2_per_page
if repeat_factor > 1:
    train_2 = pd.concat([train_2] * repeat_factor, ignore_index=True)

In [29]:
print(len(train_1),len(train_2))
print(patch_1_per_page, patch_2_per_page)

45120 1128
40 1


In [36]:
# Add a 'patch_num' column to train_1: unique number per row within each 'page' group
train_1['patch_num'] = train_1.groupby('page').cumcount()
train_2['patch_num'] = train_2.groupby('page').cumcount()
train_1['patch_num'] = train_1['patch_num'] % patch_2_per_page
merged_df = pd.merge(train_1, train_2, on=['page','patch_num'], suffixes=('_1', '_2'))

In [37]:
print(len(merged_df))
cols_to_drop = [c for c in merged_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

45120
Columns to drop: ['writer', 'isEng', 'same_text', 'file_name_1', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page', 'patch_num', 'file_name_2']


In [38]:
if 'file_name_1' in merged_df.columns and 'file_name_2' in merged_df.columns:
    all_equal = (merged_df['file_name_1'] == merged_df['file_name_2']).all()
    print(f"file_name_1 always equals file_name_2: {all_equal}")
else:
    print("file_name_1 or file_name_2 not found in merged_df columns.")

file_name_1 always equals file_name_2: True


In [39]:
# For each group of rows with the same 'page', check if all 'file_name_1' values are equal
file_name_consistency = merged_df.groupby('page')['file_name_1'].nunique()
all_equal = (file_name_consistency == 1).all()
print(f"file_name_1 is always equal within each page group: {all_equal}")
if not all_equal:
    inconsistent_pages = file_name_consistency[file_name_consistency > 1].index.tolist()
    print(f"Inconsistent pages: {inconsistent_pages}")

file_name_1 is always equal within each page group: True


In [ ]:
# For each group of rows with the same 'page', check if all 'file_name_1' values are equal
file_name_consistency = merged_df.groupby('page')['male'].nunique()
all_equal = (file_name_consistency == 1).all()
print(f"male is always equal within each page group: {all_equal}")
if not all_equal:
    inconsistent_pages = file_name_consistency[file_name_consistency > 1].index.tolist()
    print(f"Inconsistent pages: {inconsistent_pages}")

male is always equal within each page group: True


In [20]:
# Get the columns to drop (non-feature columns) for train_2
cols_to_keep = [c for c in train_2.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()]

# Select the subset where page == 0
subset = train_2[train_2['page'] == 0]

# Check if all values in each of these columns are the same
B=True
for col in cols_to_keep:
    unique_vals = subset[col].unique()
    if len(unique_vals) > 1:
        B=False
        print(f"Column {col} has multiple unique values: {unique_vals}")
    else:
        pass
    #print(f"{col}: {unique_vals} (n_unique={len(unique_vals)})")